# 2015–16 CRDC data cleaning

Step-by-step walkthrough of `scripts/clean_201516.py`. It produces the shared 50-column school table used by the other years (2017-18, 2020-21, 2021-22), so the four years can be stacked for modeling.

2015–16 is usable only after repairing Excel-corrupted IDs. Negative reserved codes are missingness, not counts: they are set to NA and never filled with 0. The original codes go to the quality log (`quality_detail_201516.csv`), not the main file.

Follows the shared format in `AI_CONTEXT.md` (sections A2–A5).

Steps:
1. Read CSVs as strings (UTF-8, fall back to cp1252); drop empty rows.
2. Pad `LEAID` / `SCHID`, rebuild `COMBOKEY`, left-join LEA data.
3. Recode reserved codes to NA (logged).
4. Build the shared columns: enrollment, seclusion, FTE, shares, `grade_band`, flags.
5. Check against the data audit and write `schools_201516.csv`, `schools_201516_analysis.csv`, the quality log, and the docs.

Restraint columns (`MECH` / `PHYS`) are deliberately not read: the EDA found they leak the outcome (AUC 0.84 on its own in 2015–16).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

import clean_201516 as cc

RAW_DIR = cc.DEFAULT_RAW_DIR      # data/raw/2015-16
OUT_DIR = cc.DEFAULT_OUT_DIR      # data/processed/201516 (not tracked by git)
DOCS_DIR = cc.DEFAULT_DOCS_DIR    # docs/cleaning

## 1. Read priority columns as strings

2015–16 files are `cp1252`. The reader tries UTF-8 first and falls back. Every value is a string so IDs keep leading zeros.

In [ ]:
sch, sch_info = cc.read_crdc_csv(RAW_DIR / cc.SCH_FILE, cc.SCH_COLS)
lea, lea_info = cc.read_crdc_csv(RAW_DIR / cc.LEA_FILE, cc.LEA_COLS)
sch.shape, lea.shape

## 2. Rebuild `COMBOKEY` and join LEA data

The stored `COMBOKEY` dropped leading zeros and used scientific notation. It is rebuilt as padded `LEAID` (7) + `SCHID` (5). The function raises if the key is not unique or if the LEA file has duplicate `LEAID`s.

In [ ]:
df, join_info = cc.build_keys_and_join(sch, lea)
join_info

## 3. Recode reserved codes

Values `< 0` become NA and each one is written to the quality log with its raw code and reason group (structural skip, missing, suppressed). Zeros stay zeros.

In [ ]:
df, recode_logs = cc.recode_reserved(df, cc.COUNT_COLS)
pd.concat(recode_logs).query("issue == 'reserved_code'").groupby(["raw_column", "raw_value"]).size().unstack(fill_value=0)

## 4. Build the shared 50 columns

- `enrollment_total` = `TOT_ENR_M` + `TOT_ENR_F` (NA if either is NA). `enr_nonbinary` is always NA (no X counts in 2015–16).
- `seclusion_instances` = WODIS + IDEA + 504, only when all three are valid. `seclusion_rate` is per 100 students and NA when enrollment is missing or 0.
- `pct_idea` / `pct_504` / `pct_el` are on a 0–100 scale and NA when the subgroup is larger than the total.
- `grade_band` is elementary / middle / high / span / other from the KG–12 flags (NA if a flag is unknown); `support_fte_per_100` = nurses + psychologists + social workers per 100 students (counselors not included).
- FTE of 999 for counselors, nurses, psychologists and social workers is a placeholder: set to NA and logged. FTE outliers are flagged, not changed.
- `flag_small_enrollment` is under 10 students. `flag_missing_enrollment` is True when enrollment is NA or not above 0. `flag_analysis_ready` needs a valid seclusion total and enrollment above 0.

In [ ]:
out, feature_logs = cc.build_features(df)
print(out.shape)
out.head()

## 5. Audit checks and export

In [ ]:
quality = pd.concat(recode_logs + feature_logs, ignore_index=True)[cc.QUALITY_COLUMNS]
result = {"work": df, "out": out, "quality": quality,
          "sources": [sch_info, lea_info], "join": join_info}

report = cc.audit_report(result)
cc.print_audit_checks(report)
assert list(out.columns) == cc.OUTPUT_COLUMNS and len(out.columns) == 50

In [ ]:
cc.write_outputs(result, OUT_DIR)
cc.write_mapping(DOCS_DIR / "mapping_201516.csv")
cc.write_qa(result, report, DOCS_DIR / "qa_201516.md")